# TB Portals — **Agentic 6-Rung Pipeline**: mode `a2`
Frozen RAD-DINO features + rungs `1 2 3 4 5 6`. Run end-to-end; download `agentic_a2.zip`.
Attach only **tb-portals-cxr-pngs**; Internet **ON**; GPU T4.

## 0 — Clone  *(restart kernel after any pull that changed .py)*

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + "/scripts"):
    if _p not in sys.path: sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)
# After a git pull that changed .py modules, RESTART the kernel so Python reloads them.

## Install deps (transformers for RAD-DINO; torchxrayvision = fallback backbone)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "torchxrayvision", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

## Paths

In [ ]:
import os
WORK          = "/kaggle/working"
REPO_DIR      = "/kaggle/working/dl-project-codebase"
DATASET       = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT = f"{DATASET}/kaggle_export"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
MODE          = "a2"
BACKBONE      = "rad-dino"      # rad-dino (primary) | txrv (fallback) | densenet (control)
PATCH_GRID    = 0          # 0 = CLS vector; >0 = GxG patch grid (a1 spatial head)
FEATURES      = f"{WORK}/features_{BACKBONE}_cls.npz"
print("MODE:", MODE, "| KAGGLE_EXPORT:", os.path.isdir(KAGGLE_EXPORT), "| features ->", FEATURES)

## 1 — Build the 5,010-image manifest (Kantipudi Table 1)

In [ ]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

## 2 — Cache frozen features (~10–20 min, one-time; idempotent)

In [ ]:
import os, sys
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
if os.path.isfile(FEATURES):
    print("features already cached ->", FEATURES)
else:
    from cache_features import main as cache_main
    cache_main(["--manifest", PAPER_MANIFEST, "--out", FEATURES,
                "--backbone", BACKBONE, "--batch-size", "32"])

## 3 — Run the rung ladder for mode `a2` (rungs 1 2 3 4 5 6, 3 seeds)

In [ ]:
from src.training.train_agentic import main as agentic_main
agentic_main(["--features", FEATURES, "--manifest", PAPER_MANIFEST,
              "--mode", MODE, "--out-dir", f"{WORK}/agentic_{MODE}",
              "--rungs", '1', '2', '3', '4', '5', '6',
              "--held-outs", "Romania", "Moldova", "Kazakhstan",
              "--seeds", "0", "1", "2"])

## 4 — Save (download → baseline_runs/agentic/)

In [ ]:
import os, shutil
src = f"{WORK}/agentic_{MODE}"
# keep the feature cache alongside results so it can be re-attached to skip caching
try: shutil.copy(FEATURES, f"{src}/{os.path.basename(FEATURES)}")
except Exception as e: print("cache copy skipped:", e)
zip_path = shutil.make_archive(f"{WORK}/agentic_{MODE}", "zip", src)
print("Saved ->", zip_path)
print("Download it; drop results_agentic_*.csv + preds_*.csv into baseline_runs/agentic/.")